# Bottlenecks in grievance disposal, July 2024 - June 25

Between July 2024 and June 2025, Odisha's Janasunani portal received roughly a little under seven lakh complaints and disposed around 70 per cent of these. For the complaints that were closed, the average time to close was 63 days. The goal of this note is to decompose the total time taken into the stops at various offices a complaint traverses during its journey from receipt to closure, and identify the bottlenecks that cause most of the hold-up.

In [ ]:
# The two complaint types the note holds fixed. Edit to change them.
SUBCATEGORIES = [
    ("rh", "Rural housing", "dept_id = 21 AND subcategory = 'Rural Housing'"),
    ("oap", "Old age and disability pensions", "dept_id = 40 AND subcategory = 'OAP/ODP/MBPY/WP'"),
]

In [ ]:
import html

import polars as pl
from IPython.display import Markdown, display

from janasunani.analytics.bottlenecks import check_note, open_lake
from janasunani.analytics.exhibits import (
    ENTRY_SHORT, draw_district_offices, draw_field_drivers, draw_field_levels,
    draw_office_spread, draw_stage_bar, draw_stage_stack,
    entry_route_groups, entry_routes, field_drivers, field_frame, field_levels,
    field_roles, office_means, office_vs_district, route_stage_roles,
    stage_means, stage_rows, stage_table,
)
from janasunani.analytics.figures import Report
from janasunani.analytics.journey import STAGE_NAMES, STAGES_THREE
from janasunani.analytics.sample import Sample

con = open_lake()
con.execute("SET enable_progress_bar = false")
pl.Config.set_fmt_str_lengths(120)
# Silent on pass: the CA&GR Analytics Note's published 2024-25 figures.
check_note(con)

# Tables are for a reader, not a debugger: no shape line, no dtype row, no
# quotes around text, blanks for nulls, thousands separators on counts. This
# also applies to tables the exhibits display, and survives export to Word.
def _cell(v):
    if v is None:
        return ""
    return f"{v:,}" if isinstance(v, int) else html.escape(str(v))


def _table_html(df):
    head = "".join(f"<th>{html.escape(c)}</th>" for c in df.columns)
    body = "".join("<tr>" + "".join(f"<td>{_cell(v)}</td>" for v in row) + "</tr>"
                   for row in df.rows())
    return f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"


def _table_md(df):
    rows = [list(df.columns), ["---"] * df.width,
            *[[_cell(v) for v in row] for row in df.rows()]]
    return "\n".join("| " + " | ".join(map(str, r)) + " |" for r in rows)


_fmt = get_ipython().display_formatter.formatters
_fmt["text/html"].for_type(pl.DataFrame, _table_html)
# The Word version is built from a Markdown export, which reads this one.
_fmt["text/markdown"].for_type(pl.DataFrame, _table_md)


STEPS = [
    ("F0", "Complaints filed in 2024-25", "TRUE"),
    ("F1", "Disposed", "is_disposed"),
    ("F2", "Disposed, with a usable closing date",
     "is_disposed AND days_to_close IS NOT NULL"),
]
FINAL = "Final sample: every office on the route identified, stage dates consistent"
FIGDIR = "../outputs/figures"
D2 = "Stage 2 days"
# The field-action part stands on complaints filed July to December 2024: a
# slow complaint filed late in the year is often still open.
H1 = "created_on < DATE '2025-01-01'"
# How each entry office reads inside a sentence.
IN_SENTENCE = {"Collector": "the Collector", "CM Cell": "the CM Cell",
               "Department, directly": "direct filing with the department",
               "Chief Secretary": "the Chief Secretary", "Police": "the police",
               "Governor's office": "the Governor's office",
               "Entry not recorded": "no recorded entry point"}


def sample(name, where):
    s = Sample(con, ns=f"{name}_", base=f"year = '2024-25' AND {where}", steps=STEPS)
    s.install_route("F2", "F2")
    s.install_phases("F2", "F2")
    s.STEPS["P4"] = (FINAL, None)
    return s


def funnel(s):
    display(s.funnel_table(["F0", "F1", "F2", "P4"])
            .drop("Step").rename({"Grievances": "Complaints"}))


def pct(a, b):
    return f"{100 * a / b:.0f}%"


def decomposition(s, rep, label):
    means, total = stage_means(s, STAGES_THREE)
    s2 = float(means["Stage 2"])
    display(Markdown(
        f"**A disposed complaint takes {total:.0f} days on average. Stage 2 takes "
        f"{s2:.0f} of them ({pct(s2, total)}).**"))
    draw_stage_bar(rep, means, total, f"Time to disposal by stage, {label}",
                   f"{s.FUNNEL['P4']:,} disposed complaints. Averages; the three "
                   f"stages add up to {total:.0f} days.", "02_stages", STAGES_THREE)
    t = stage_table(s, total, STAGES_THREE).rename({"Phase": "Stage"})
    display(t.with_columns(pl.col("Stage").replace_strict(
        {k: f"{k}: {v[0]}" for k, v in STAGE_NAMES.items()})))


def entry_and_routes(s, rep, label):
    t = entry_routes(s, top=3)
    rows = list(t.iter_rows(named=True))
    names = [IN_SENTENCE[ENTRY_SHORT[r["Entry"]]] for r in rows]
    enter = ", ".join(f"{n} ({r['% of sample']:.0f}%)" for n, r in zip(names[:-1], rows[:-1]))
    enter += f" and {names[-1]} ({rows[-1]['% of sample']:.0f}%)"
    routes = "; from ".join(f"{n}, {r['Commonest route']} ({r['% of entry']:.0f}%)"
                            for n, r in zip(names, rows))
    display(Markdown(
        f"**Most complaints enter through {enter}.** The most common route "
        f"from {routes}."))
    groups = entry_route_groups(t)
    split = route_stage_roles(s, groups, STAGES_THREE)
    braces = [{k: r[k] for k in STAGES_THREE} for r in split.iter_rows(named=True)]
    stack = stage_rows(s, groups, STAGES_THREE, floor=1)
    draw_stage_stack(rep, stack, "Entry and route", f"Time to disposal by stage, top route per entry office, {label}",
                     "Averages. The braces name the offices that hold the complaint in each stage.",
                     "03_routes", STAGES_THREE, share_of=s.FUNNEL["P4"], braces=braces,
                     count_label="Complaints")


def show_field(s, rep, label, floor=30, spread=True):
    frame = field_frame(s)
    display(Markdown(
        f"**{frame.height:,} complaints filed July to December 2024 reached Stage 2.** "
        f"On average they spent {frame['field_action'].mean():.0f} days in it."))
    levels, roles = field_levels(s, frame), field_roles(s, frame)
    draw_field_levels(rep, levels, roles, f"Stage 2 days by office level, {label}",
                      "Each day is counted at the office holding the complaint, "
                      "so the parts add up to the average.", "04_field_levels", days_label=D2)
    per_office = frame.height / frame["field_office"].n_unique()
    if per_office >= floor:
        shares, full = field_drivers(frame)
        draw_field_drivers(rep, shares, full, f"Share of the variation in Stage 2 days, by factor, {label}",
                           f"{frame.height:,} complaints.", "05_field_drivers", days_label=D2)
    else:
        display(Markdown(
            f"*Not ranked here: with about {per_office:.0f} complaints per office, an "
            "office's effect cannot be told from chance. See the statewide ranking.*"))
    role = frame["field_role"].mode().sort()[0]
    if spread:
        subcat = frame.filter(pl.col("field_role") == role)["subcategory"].mode().sort()[0]
        same = office_means(frame, floor, role=role, subcategory=subcat)
        draw_office_spread(rep, same, f"Stage 2 days by {role} office, {label}",
                           f"Every {role} office with at least {floor} complaints. "
                           "Average Stage 2 days per office.", "06_office_spread", days_label=D2)
        display(Markdown(f"**Fastest and slowest five {role} offices**"))
        display(pl.concat([same.head(5), same.tail(5)]).rename(
            {"Claims": "Complaints", "Mean field days": "Average Stage 2 days"}))
    offices = office_means(frame, floor, role=role)
    big = (frame.filter(pl.col("field_role") == role).group_by("district").len()
           .sort("len", descending=True).head(6)["district"].to_list())
    within, between = office_vs_district(offices, big)
    display(Markdown(
        f"In these districts, offices in the same district typically differ by "
        f"{within:.0f} days; district averages differ by {between:.0f} days."))
    draw_district_offices(rep, offices, big, f"Stage 2 days by {role} office, six largest districts, {label}",
                          f"{role} offices with at least {floor} complaints, in the "
                          "six districts with the most.", "07_district_offices",
                          days_label=D2)


## Data and sample

All complaints filed between July 2024 and June 2025. Times are measured on disposed complaints whose records date every stage.

In [ ]:
state = sample("state", "TRUE")
assert state.FUNNEL["F0"] == 688_301, state.FUNNEL["F0"]   # the published total
funnel(state)

## Three stages

| Stage | | Covers |
|---|---|---|
| **Stage 1** | Processing | From receipt to route selection |
| **Stage 2** | Field action | Action taken for disposal |
| **Stage 3** | Review | Review by the assigning authority |

Stage 3 is zero when a complaint is not sent back for review. The time from the last action to formal closure averages under a day and is left out.

## Time to disposal

In [ ]:
rep = Report(con, [("Statewide", "TRUE")], FIGDIR, prefix="disp_state_")
decomposition(state, rep, "statewide")

<div style="page-break-before: always"></div>

## By complaint type

Some complaints need more work than others. To compare like with like, the rest of this part follows two common complaint types.

In [ ]:
subs = {key: (label, sample(key, where)) for key, label, where in SUBCATEGORIES}

In [ ]:
label, s = subs["rh"]
display(Markdown(f"## {label}\n\n**{s.FUNNEL['F0']:,} complaints were filed; "
                 f"{s.FUNNEL['P4']:,} are in the final sample.**"))
rep = Report(con, [(label, "TRUE")], FIGDIR, prefix="disp_rh_")
decomposition(s, rep, label.lower())

In [ ]:
entry_and_routes(subs["rh"][1], Report(con, [(subs["rh"][0], "TRUE")], FIGDIR, prefix="disp_rh_"), subs["rh"][0].lower())

In [ ]:
label, s = subs["oap"]
display(Markdown(f"## {label}\n\n**{s.FUNNEL['F0']:,} complaints were filed; "
                 f"{s.FUNNEL['P4']:,} are in the final sample.**"))
rep = Report(con, [(label, "TRUE")], FIGDIR, prefix="disp_oap_")
decomposition(s, rep, label.lower())

In [ ]:
entry_and_routes(subs["oap"][1], Report(con, [(subs["oap"][0], "TRUE")], FIGDIR, prefix="disp_oap_"), subs["oap"][0].lower())

## Summary

*Placeholder for the closing paragraph.*

- Where the time goes: [Stage 2 share, statewide and in both complaint types]
- What differs by entry office and route: [the slowest route and its stage]
- What a Secretary could act on: [one or two levers]
- What the data cannot tell us: [open complaints untimed; no reasons recorded]

<div style="page-break-before: always"></div>

# Stage 2 in detail

Stage 2 takes most of the time. This part uses complaints filed July to December 2024, because slow complaints filed later are often still open. It shows:

1. **Stage 2 days by office level and by office.** Each day is counted at the office holding the complaint.
2. **Share of the variation in Stage 2 days, by factor.** Complaints are compared on one factor at a time (the field office, the office that sent the complaint down, the route, the complaint type, the district) and each factor's share of the explained difference is measured. Where two factors overlap, such as an office and its district, the shared part is split evenly.
3. **Stage 2 days by office, for one complaint type and within districts.**

## Statewide

In [ ]:
show_field(sample("state_h1", H1), Report(con, [("Statewide", "TRUE")], FIGDIR, prefix="disp_state_"), "statewide", spread=False)

In [ ]:
key, label, where = next(x for x in SUBCATEGORIES if x[0] == "rh")
display(Markdown(f"## {label}"))
show_field(sample(f"{key}_h1", f"{where} AND {H1}"), Report(con, [(label, "TRUE")], FIGDIR, prefix=f"disp_{key}_"),
           label.lower(), floor=30 if key == "rh" else 15)

In [ ]:
key, label, where = next(x for x in SUBCATEGORIES if x[0] == "oap")
display(Markdown(f"## {label}"))
show_field(sample(f"{key}_h1", f"{where} AND {H1}"), Report(con, [(label, "TRUE")], FIGDIR, prefix=f"disp_{key}_"),
           label.lower(), floor=30 if key == "rh" else 15)

## Limitations

- **Open complaints are not timed**, so recent periods look faster than they will turn out.
- **A long stage is not idle time.** It may be field enquiry, a statutory wait, or waiting on the citizen.
- **Repeat filings count separately.** A citizen who filed three times counts three times.

In [ ]:
con.close()